# Model Validation (Second Covariate Analysis) [Version updated: 07-07-2026]

This notebook guides you through selecting and validating climate models (CMIP6 and CORDEX) for use in the attribution study.

---

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervention </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  

---

## What This Notebook Does

This notebook helps you:
- Load CMIP6 and CORDEX data (daily mean Tmax, Tmean, Tmin, precipitation)
- Validate models based on:
  - Seasonal cycle shape
  - Climatological spatial pattern
  - Distribution fit parameters
- Decide which models to keep for Step 5 (Attribution)

---


### <font color="green">  **Imports** </font>
**Before proceeding, ensure you have executed the cell below.**


In [ ]:
# Imports
from c3s_event_attribution_tools import *
import gc
import os 
from datetime import datetime, timedelta
import geopandas as gpd
import xarray as xr
from pathlib import Path
import cartopy.crs as ccrs
import ipywidgets as widgets
import json
from IPython.display import display, clear_output
import pandas as pd
from typing import Dict, Any
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri
from rpy2.robjects.vectors import ListVector
# import R libraries from WWA
ro.r('''
if (!requireNamespace("remotes", quietly = TRUE)) {
    install.packages("remotes", repos="https://cloud.r-project.org", quiet=TRUE)
}

# Force install the older, compatible version of gsl
if (!requireNamespace("gsl", quietly = TRUE)) {
    remotes::install_version("gsl", version = "2.1-8", repos = "https://cloud.r-project.org")
}

# Now install rwwa (Maris fork to ensure availablity)
remotes::install_github("maris-development/rwwa@236d9a6b4a201eca1f28001b5535341022f5aeaf") 
''')

rwwa = importr("rwwa")
%load_ext rpy2.ipython
from rpy2.robjects import r, globalenv
import rpy2.robjects.pandas2ri as pandas2ri

### <font color="orange"> **User specifications** </font>

#### <font color="orange"> Authentication & File Setup </font>

Action Required: Ensure you have entered your **CDS API Key** and **CORDEX ARCO Token** in the code cell above.

Storage: Data and results will be saved to: {{your_save_directory}}.

Security: ⚠️ Never share this notebook publicly or commit it to GitHub with your API keys visible.

Next Step: Once keys are set, run the cell to initialize the DataClient for cloud data access.

In [ ]:
# Api key used for the DataClient, replace with your own API key from the C3S Climate Data Store
################# (User selection) ###################
your_api_key = ''
cordex_arco_token = ''
######################################################

# Initialize DataClient
data_client = DataClient(
    cds_key=your_api_key,
    cordex_arco_token=cordex_arco_token
)

# Do not touch
CURRENT_DIRECTORY = os.getcwd() 

# Directory you wish to store output files in, using ../ specifies the parent directory
################# (User selection) ###################
your_save_directory = os.path.abspath(os.path.join(CURRENT_DIRECTORY, "./data"))   # change ./data to your desired directory
######################################################

# Create directory if it does not exist
os.makedirs(your_save_directory, exist_ok=True) 

# Decision table file path
validation_file_path = os.path.join(your_save_directory, "res-models-validation.csv")

# Load state validation results and active parameters if they exist, otherwise initialize empty variables
try:
    df_validation_results = pd.read_csv(validation_file_path)
    with open(os.path.join(your_save_directory, "active-params.json"), 'r') as f:
        active_params = json.load(f)
    print(f"Existing validation results loaded from {validation_file_path}. Now you can modify your taken decisions by running ONLY the Validation Table Cells you want to modify. Your decisions are overwritten if you run cell 4.1.a")
except FileNotFoundError: 
    print("No existing validation results or active parameters found. A new file will be created at the end of this notebook.")

#### <font color="orange"> Event Definition & Study Area </font>
Action Required: Define the core parameters for your attribution study. These settings must match the "Event Definition" established in previous steps 2.1 and 2.2.

Bounding Box (bbox): Define the spatial extent of the event (South, West, North, East).

Parameter: Select the variable to analyze (Tmean, Tmax, Tmin, or Precipitation).

Event Date: Set the specific date of the extreme event.

Required Files: Ensure the study region shapefile and observed climatology/seasonal cycle datasets are present in your save directory.

In [ ]:
################################## (User selection) ##############################################
# Area of interest bounding box:
# (deg Southern boundary, Western boundary, Northern boundary, Eastern boundary)
bbox = (, , , )

# Choice of parameter (Tmax, Tmean, Tmin, Precipitation)
parameter = ''

# Date of the event
event_end = datetime(, , ) #YYYY,MM,DD
# Calculate event start date (14 days before the event end date)
event_start = event_end - timedelta(days=14)

##################################################################################################
config = Utils.get_parameter_config(parameter)
variable = config["variable"]

## Load study region and data from Event Definition step
# Load study region shapefile
study_region = gpd.read_file(os.path.join(your_save_directory, "sf_studyregion.shp"))

# Load observed seasonal cycle data
obs_seasonal_cycle = xr.open_dataset(os.path.join(your_save_directory, "seasonal_cycle_1991-2020.nc"))

# Month observed climatology data
obs_climatology = xr.open_dataset(os.path.join(your_save_directory, f"{variable}_monthly_climatology.nc"))

# Annual observed climatology data
obs_climatology_annual = xr.open_dataset(os.path.join(your_save_directory, f"climatology_1991-2020_{variable}.nc"))

bbox = Utils.convert_bbox(*bbox)
cordex_p = Utils.var_map(parameter, "cordex")
cmip6_p = Utils.var_map(parameter, "cmip6")

# Get parameter configuration
config = Utils.get_parameter_config(parameter)
variable = config["variable"]
y_label = config["y_label"]
unit = config["unit"]
calculation = config["calculation"]  # absolute or relative
method = config["method"] 
from_unit = config.get("from_unit", "k")
to_unit= config.get("to_unit", "c")
value_col = config.get("value_col", "value")

# Observed climatology
obs_climatology = obs_climatology.set_index(index=['latitude', 'longitude']).unstack('index')
obs_climatology = obs_climatology[value_col]
obs_climatology_annual = obs_climatology_annual[value_col].mean(dim="dayofyear")

# Fit type configuration for RWWA
if parameter in ["Tmax", "Tmean", "Tmin"]:
    conf = "shift"
elif parameter == "Precipitation":
    conf = "fixeddisp"

## <font color="orange"> 4. List of models </font>

#### <font color="orange"> CMIP6 Selection </font>
**Action Required:** Select the Global Climate Models (GCMs) and time ranges for the CMIP6 ensemble.

- **Model Selection (GCM_models):** Default use all available models. Enter the short names of the CMIP6 models you wish to evaluate (e.g., access_cm2, canesm5).

- **Max Models:** In case of quick testing, you can use max_models to set a maximum number of models to download. For official analyses keep max_models = None to download all models available

- **Time Ranges:** Default use all years. Define the historical_time_range (baseline) and future_time_range (projections). Per protocol, the future range typically extends to 2100 to capture high-warming scenarios.

- **Experiment:** The default is set to SSP5-8.5 and Historical to represent the full range of forced climate response.

Next Step: Run this cell to fetch the CMIP6 daily data and the corresponding Global Mean Surface Temperature (GMST). The DataClient will automatically pair the climate data with the the corresponding data needed to calculate the GMST required for the non-stationary GEV fitting in Step 5.

In [ ]:
############################################### (User selection) ##############################################
# Limit the maximum number of models to download
max_models = 1  # Set to None for no limit, or specify an integer (e.g., max_models = 5)
# Future and historical time ranges
future_time_range = (datetime(2015, 1, 1), datetime(2100, 12, 31))
historical_time_range = (datetime(1880, 1, 1), datetime(2014, 12, 31))
# Experiments
experiment = ["ssp5_8_5", "historical"]
###############################################################################################################

#### <font color="orange"> CORDEX Selection </font>

##### <font color="green"> CORDEX domains </font>

Next Step: Run these cells to visualize the domain coverage and build the list of available CORDEX models. The output will show the total number of models found in the catalog that match your selection. This cell automatically compares your Bounding Box and Study Region against CORDEX boundaries to suggest the most suitable domain.

In [ ]:
# Process CORDEX domains
mapproj = ccrs.PlateCarree()
domains_data = Utils.get_cordex_domain_configs()
gdf = Utils.create_cordex_gdf(domains_data, mapproj) # Keep this for the plot
selected = Utils.find_covering_domain(domains_data, study_region, bbox)
print(f"Recommended Domain: {selected} ({domains_data[selected]['long_name']})" if selected else "No single domain fully covers the study area.")
# Plot CORDEX domains
fig, ax, img_ax = Plot.plot_cordex_map(gdf, domains_data, bbox, study_region, mapproj, selected_domain=selected, buffer=10)

##### <font color="orange"> Action Required: </font>

Identify and select the appropriate Regional Climate Model (RCM) domain for your study area. Confirm your selection by entering the domain code (e.g., ["EUR"], ["AFR"], or ["NAM"]). Additionally, if you want to filter by resolution, add it next to the domain name to download only the selected resolution (e.g., ["EUR11"], ["AFR22"], or ["NAM44"]).


Due to data retrieval speeds issues, at the moment only the EUR domain can be used.

- **Max Models:** In case of quick testing, you can use max_models to set a maximum number of models to download. For official analyses keep max_models = None to download all models available

Model Pairing: The script scans the cloud catalog to find all available GCM-RCM pairings for that specific domain.

In [ ]:
############ (User selection) ####################
# Select the CORDEX domain to use for the analysis 
domain = [""]

# Limit the maximum number of models to download
max_models_ = 1  # Set to None for no limit, or specify an integer (e.g., max_models_ = 5)
# Future and historical time ranges
historical_time_range_ = (datetime(1950, 1, 1), datetime(2005, 12, 31))
future_time_range_ = (datetime(2006, 1, 1), datetime(2100, 12, 31))
##################################################

catalog_path = Path("./00_BASE_DATA/projections_cordex_domains_single_levels-ZARR-Access-new.txt").expanduser()
assert catalog_path.exists(), f"Catalog not found: {catalog_path.resolve()}"

cordex_models = Process.build_cordex_model_pairs(catalog_path, domain)
print(len(cordex_models), 'CORDEX models found for domain', domain, "with max models set to", max_models_)

---

## 4. Model Validation Procedure

Each model or model ensemble needs to undergo a validation procedure. Expert judgement is necessary to decide which models are scientifically sound enough to include in the final synthesis. Please use the validation table to track your decisions; your entries and comments will remain active and editable throughout the process. You can modify your decisions at any time by editing the corresponding cells within the table. 

- When adding new comments, please merge them to your previous entries—separating them with a comma or hyphen. 
- If you have updated decisions or comments in other sections, remember to rerun this cell to synchronize the view and display the latest updates in the validation table.
- Do not change 'False' back to 'True' unless you have a good reason - the 'Include' column is general and not per validation measure

**Important:** If the validation table does not show up, reload the browser.

**Important:** You must click the "💾 Save" button to persist your decisions to disk and update the final results table.

### 4.1 General Properties

Evaluate the model's structural ability to simulate the event by assessing resolution, seasonal cycle, and spatial pattern against the reference observational product.

#### <font color = 'orange'> 4.1.a Model resolution vs event spatial scale </font>

Make an initial decision on which model ensembles (CMIP6, CORDEX) to use based on model resolution vs event spatial scale. Is the resolution of the model sufficient to represent the event and for comparison with observations?  

i. For precipitation, in case of convective precipitation events:  
- 1. If the event is large scale but convection is clearly present (this is the case if a go-decision has been reached based on Step 1.8a.i case 4), omit CMIP6 and use only CORDEX. Write this decision in the Scientific report section 2.2. 
- 2. If the event is large scale but convection plays a major role (e.g. organised convection in thunderstorms), no attribution is possible without convection permitting models and we can’t use CMIP6 or CORDEX. This situation has probably already led you to a no-go decision in Step 1.8a 

ii. For precipitation, if the event happens close to an orographic barrier, omit CMIP6. 

iii. Else, use both CORDEX and CMIP6 

Please select the models you want to analyse based on the previous statements using **'True'** or **'False'**, then run the cell to download the corresponding models and create the validation table


In [ ]:
######################## (User selection) #############################################
Run_CMIP6 = True  
Run_CORDEX = True
#######################################################################################

if Run_CMIP6:
    GCM_models = Utils.get_cmip6_models(parameter)
    result = data_client.fetch_climate_scenarios(
        analysis_type='cmip6',
        models=GCM_models,
        variable_name=cmip6_p,
        bbox=bbox,
        study_region=study_region,
        hist_range=historical_time_range,
        fut_range=future_time_range,
        temp_res="daily",
        max_models=max_models,
        second_cov=True
    )
    CMIP6_models, CMIP6_GMST, CMIP6_SST = result


if Run_CORDEX:
    result = data_client.fetch_climate_scenarios(
        analysis_type='cordex',
        models=cordex_models,
        variable_name=cordex_p,
        bbox=bbox,
        study_region=study_region,
        hist_range=historical_time_range_,
        fut_range=future_time_range_,
        temp_res="daily",
        max_models=max_models_,
        second_cov=True
    )
    cordex_models, cordex_GMST, cordex_SST = result

# Create a list to store validation
validation_list = []

# Extract successfully loaded CMIP6 models
if Run_CMIP6:
    for model_id in CMIP6_models.keys():
        validation_list.append({'model': model_id,'project': 'CMIP6', 'Seasonal cycle': '', 'Spatial maps': '', 'sigma_validation' : '', 'shape_validation' : '', 'disp_validation' : '', 'validation_summary' : '', 'Stat Fit': 'Pending',
        'Comments': '', 'Include T/F' : ''
        })

# Extract successfully loaded CORDEX models
if Run_CORDEX:
    for model_id in cordex_models.keys():
        validation_list.append({'model': model_id, 'project': 'CORDEX', 'Seasonal cycle': '', 'Spatial maps': '', 'sigma_validation' : '', 'shape_validation' : '', 'disp_validation' : '', 'validation_summary' : '', 'Stat Fit': 'Pending',
        'Comments': '', 'Include T/F' : ''
        })

# Create the DataFrame
df_validation_results = pd.DataFrame(validation_list)

####  <font color = 'orange'> 4.1.b. Seasonal Cycle </font>
Does the model seasonal cycle resemble the observed seasonal cycle?

##### <font color = 'green'> Calculations </font>

Run the next cell to compute the corresponding calculations for seasonal cycle, spatial pattern and annual time series. This cell might take a while to finish

In [ ]:
if Run_CORDEX:
    CORDEX_analysis = Process.compute_climate_indices(
        data_input=cordex_models,
        parameter=cordex_p,
        study_region=study_region,
        baseline_range=("1991", "2020"),
        sc_month_range=(1,12),
        clim_month_range=(event_start.month, event_end.month)
    )

if Run_CMIP6:
    CMIP6_analysis = Process.compute_climate_indices(
        data_input=CMIP6_models,
        parameter=cordex_p,
        study_region=study_region,
        baseline_range=("1991", "2020"),
        sc_month_range=(1,12),
        clim_month_range=(event_start.month, event_end.month)
    )   

##### <font color = 'orange'> Cordex Seasonal Cycle </font>
Expert Judgement Criteria:

i. Timing: do the peaks in the model seasonal cycle correspond to the peaks in observations (in timing)?  

ii. Categorization: Note seasonal cycle based on the following: 
- **“good”** peaks resemble the observed peaks
- **“reasonable”** peaks resemble most important features but there are some differences  
- **“bad”** peaks do not resemble observed peaks

iii. Write findings and decisions in table

In [ ]:
if Run_CORDEX:
    title = f"{variable} Seasonal Cycle ({unit}) - 1991 to 2020"
    legend_title = f"{variable} ({unit})"
    fig, ax, img_ax = Plot.plot_seasonal_cycles(CORDEX_analysis["seasonal_cycles"], obs_seasonal_cycle, value_col=value_col, legend_title=legend_title, title=title)

In [ ]:
# Validation Table update
Utils.create_decision_hub(df_validation_results, step='seasonal', project_filter='cordex', save_path=validation_file_path)

##### <font color = 'orange'> CMIP6 Seasonal Cycle </font>
Expert Judgement Criteria:

i. Timing: do the peaks in the model seasonal cycle correspond to the peaks in observations (in timing)?  

ii. Categorization: Note seasonal cycle based on the following: 
- **“good”** peaks resemble the observed peaks
- **“reasonable”** peaks resemble most important features but there are some differences  
- **“bad”** peaks do not resemble observed peaks

iii. Write  findings and decisions in table

In [ ]:
if Run_CMIP6:
    title = f"{variable} Seasonal Cycle ({unit}) - 1991 to 2020"
    legend_title = f"{variable} ({unit})"
    fig, ax, img_ax = Plot.plot_seasonal_cycles(CMIP6_analysis["seasonal_cycles"], obs_seasonal_cycle, value_col=value_col, legend_title=legend_title, title=title)

In [ ]:
# Validation Table update
Utils.create_decision_hub(df_validation_results, step='seasonal', project_filter='cmip6', save_path=validation_file_path)

#### <font color = 'orange'> 4.1.c. Spatial Pattern </font>

Does the spatial pattern of the variable resemble the observed spatial pattern?



##### <font color = 'orange'> Cordex Spatial Pattern </font>

i. Expert Judgement Criteria Categorization:

- **Good:** Main peaks and troughs overlapping in location with observed peaks and troughs, and relative amplitudes of features are ok
- **Reasonable:** Most of the main features are captured
- **Bad:** Few of the main features are captured.

ii.	Write findings and decisions in table.

In [ ]:
if Run_CORDEX:
    title = f"{variable} Climatology ({unit}) - 1991 to 2020"
    fig, axs, img_ax = Plot.plot_spatial_maps(obs_climatology, CORDEX_analysis["spatial_maps"], value_col=value_col, study_region=study_region, show_study=True, legend_title=title)

In [ ]:
# Validation Table update
Utils.create_decision_hub(df_validation_results, step='spatial', project_filter='cordex', save_path=validation_file_path)

##### <font color = 'orange'> CMIP6 Spatial Pattern </font>

i. Expert Judgement Criteria Categorization:

- **Good:** Main peaks and troughs overlapping in location with observed peaks and troughs, and relative amplitudes of features are ok
- **Reasonable:** Most of the main features are captured
- **Bad:** Few of the main features are captured.

ii.	Write findings and decisions in table.

In [ ]:
if Run_CMIP6:
    title = f"{variable} Climatology ({unit}) - 1991 to 2020"
    fig, axs, img_ax = Plot.plot_spatial_maps(obs_climatology, CMIP6_analysis["spatial_maps"], value_col=value_col, study_region=study_region, show_study=True, legend_title=title)

In [ ]:
# Validation Table update
Utils.create_decision_hub(df_validation_results, step='spatial', project_filter='cmip6', save_path=validation_file_path)

### 4.2	Decision model ensembles 

Write down in the scientific report Section 2.2 your final decision on which model ensembles (CMIP6, which regional CORDEX domain) are considered further for analysis. 

### 4.3 Model Time Series


##### <font color='orange'> Parameters and calculation </font>

Choose the same parameters used in Step 1. Event Definition. The yearly time series are calculated based on the selected parameters

In [ ]:
################### User Selection  ##################
# Choose mean, max, min
yearly_value = 'max'

# Padding >= 1 : rolling window, n-days (centered)
padding = 1

# Choose month range, e.g. (1, 12) or (12, 1)
month_range = (, )

# Standard method for temperature: mean and for precipitation: sum, if not specified
# If you want to change it, specify here
method = None
#######################################################

if parameter in ["Tmax", "Tmean", "Tmin", "tas"]:
    title = f"Annual time series {variable.capitalize()} ({unit}) - Observational data"
    if method is None:
        method = 'mean'
elif parameter == "Precipitation":
    title = f"Annual time series {variable.capitalize()} ({unit}) - Observational data"
    if method is None:
        method = 'sum'

if Run_CORDEX:
    cordex_yearly_series = Process.calculate_yearly_value_xr(
        time_series=CORDEX_analysis['time_series'], 
        yearly_value=yearly_value, 
        padding=padding,
        month_range=month_range,
        method=method
    )
if Run_CMIP6:
    cmip6_yearly_series = Process.calculate_yearly_value_xr(
        time_series=CMIP6_analysis['time_series'], 
        yearly_value=yearly_value, 
        padding=padding,
        month_range=month_range,
        method=method
    )

### 4.4 Model GMST (4-year smoothed)

##### <font color='orange'> Calculations </font>

Run the next cell to compute the corresponding GMST covariates for the different models available

In [ ]:
# Assuming CMIP6_GMST is your dictionary of global 'tas' datasets
event_year = event_end.year
if Run_CORDEX:
    cordex_gmst_yearly = Process.compute_gmst_anomalies(
        gmst_dict=cordex_GMST,
        event_year=event_year,
        year_range=(1850, 2100)
    )
if Run_CMIP6:
    cmip6_gmst_yearly = Process.compute_gmst_anomalies(
        gmst_dict=CMIP6_GMST,
        event_year=event_year,
        year_range=(1850, 2100)
    )

### 2nd Covariate - ENSO

In [ ]:
############## User Selection  ##############
# Make sure to match your selections with the ones in the Trend Analysis notebook
target_month   = 1
rolling_window = 3
center = True
nino_year_shift = 0
###############################################

# Load observational covariates saved in Trend Analysis (year, value, gmst, nino)
nino_obs = pd.read_csv(os.path.join(your_save_directory, "cov_observations.csv"))

# Observed current ENSO
nino_obs_now = nino_obs['nino'].iloc[-1]
print(f"Observed current ENSO (Nino3.4) = {nino_obs_now:.3f} for year {nino_obs['year'].iloc[-1]}")

# Observed ENSO amplitude (std) and the full range of available Niño years
obs_std = nino_obs['nino'].std()
rescale_range = (int(nino_obs['year'].min()), int(nino_obs['year'].max()))

print(f"Observed Niño std = {obs_std:.3f} over {rescale_range[0]}–{rescale_range[1]}")

if Run_CMIP6:
    cmip6_enso = Process.compute_enso_index(CMIP6_SST, target_month=target_month,
                                            rolling_window=rolling_window, center=center,
                                            nino_year_shift=nino_year_shift, obs_std=obs_std,
                                            rescale_range=rescale_range)
if Run_CORDEX:
    cordex_enso = Process.compute_enso_index(cordex_SST, target_month=target_month,
                                            rolling_window=rolling_window, center=center,
                                            nino_year_shift=nino_year_shift, obs_std=obs_std,
                                            rescale_range=rescale_range)

### 4.5 Statistical properties and probabilistic attribution


#### <font color='green'> a. Parameters </font>

After running this cell, you will load the annual time series from observations. Additionally, some neccesary computations are run as conversions and setting up specific parameters



In [ ]:
annual_timeseries_load = 'ts_ann_studyregion.nc'
ts_ann_studyregion = xr.open_dataset(os.path.join(your_save_directory, annual_timeseries_load))

ts_ann_studyregion = ts_ann_studyregion[value_col].to_dataframe().reset_index()
ts_ann_studyregion['year'] = pd.to_datetime(ts_ann_studyregion['year'].astype(int), format='%Y')

# Convert annual time series to dataframes
if Run_CORDEX:
    cordex_model_dfs = Utils.convert_annual_series_to_dfs(cordex_yearly_series)
if Run_CMIP6:
    cmip6_model_dfs = Utils.convert_annual_series_to_dfs(cmip6_yearly_series)

# Setting up the parameter specific variables
if parameter in ["Tmax", "Tmean", "Tmin", "tas"]:
    method = "std" 
elif parameter == "Precipitation": 
    method = "dispersion"


#### <font color='green'> b.	Scale and dispersion </font>

Plot and visually check the scale or dispersion parameter of the annual maximum time series. Add a sentence on caution to the scientific report Section 4
Plot and visually check the scale or dispersion parameter of the annual maximum time series over the full time range. Add a few words to the Comments row in the Table in the Jupyter Notebook if the scale or dispersion parameter is not stable, e.g., if there is a clear trend or a jump in the scale or dispersion parameter. Add a sentence on caution or restriction of years to the scientific report Section 4 if the problem is more general for several models.

In [ ]:
print(f"Calculating 15-year rolling window test using method: {method}")
obs_ann_15ym = Process.calculate_rolling_window(gdf=ts_ann_studyregion, value_col=value_col, datetime_col="year",
                                  window=15, method=method, min_periods=1, centering=True, ci=0.95)

cmip6_model_dfs_15ym = {}
cordex_model_dfs_15ym = {}
if Run_CMIP6:
    for name, df in cmip6_model_dfs.items():
        res = Process.calculate_rolling_window(gdf=df, value_col='value', datetime_col="year",
                                    window=15, method=method, min_periods=1, centering=True, ci=0.95)
        res['year'] = pd.to_datetime(res['year'], format='%Y')
        cmip6_model_dfs_15ym[name] = res
if Run_CORDEX:
    for name, df in cordex_model_dfs.items():
        res = Process.calculate_rolling_window(gdf=df, value_col='value', datetime_col="year",
                                    window=15, method=method, min_periods=1, centering=True, ci=0.95)
        res['year'] = pd.to_datetime(res['year'], format='%Y')
        cordex_model_dfs_15ym[name] = res
    

##### <font color='green'> CORDEX yearly time series </font>

In [ ]:
if Run_CORDEX:
    fig, axs, img_ax = Plot.plot_rolling_window_comparison(
        model_dfs=cordex_model_dfs_15ym, 
        obs_df=obs_ann_15ym, 
        value_col=value_col,
        legend_title=f"15-Year Running {method} of {parameter}",
        yaxis_label=method
    )

In [ ]:
Utils.create_decision_hub(df_validation_results, step='visual', project_filter='cordex', save_path=validation_file_path)

##### <font color='green'> CMIP6 yearly time series </font>

In [ ]:
if Run_CMIP6:
    fig, axs, img_ax = Plot.plot_rolling_window_comparison(
        model_dfs=cmip6_model_dfs_15ym, 
        obs_df=obs_ann_15ym, 
        value_col=value_col,
        legend_title=f"15-Year Running {method} of {parameter}",
        yaxis_label=method
    )

In [ ]:
Utils.create_decision_hub(df_validation_results, step='visual', project_filter='cmip6', save_path=validation_file_path)

#### 5. Probabilistic Attribution

This section details the probabilistic attribution analysis. While only models that pass the validation in Step 4 are included in the final report, it is more efficient to perform validation, attribution and projection analysis on all models at once. Therefore, Step 5 is executed here to generate results for all models and scenarios, providing the necessary data to complete the statistical validation and selection criteria in Step 4.5.

**The code below does not contain step numbers, but they do cover Steps 5.1 - 5.3.**

##### <font color="green"> Initialize functions and R dataframes </font>

In [ ]:
# Initialize empty dictionaries for all models
combined_model_dfs = {}
combined_gmst_dfs = {}
combined_enso_dfs = {}

# Add CORDEX if enabled
if Run_CORDEX:
    for key, df in cordex_model_dfs.items():
        combined_model_dfs[key] = df
        combined_gmst_dfs[key] = cordex_gmst_yearly[key]
        combined_enso_dfs[key] = cordex_enso[key]

# Add CMIP6 if enabled (merging into the same dicts)
if Run_CMIP6:
    for key, df in cmip6_model_dfs.items():
        combined_model_dfs[key] = df
        combined_gmst_dfs[key] = cmip6_gmst_yearly[key]
        combined_enso_dfs[key] = cmip6_enso[key]

# Convert the unified dictionaries to R objects once
combined_models_r = {}
combined_gmst_r = {}
combined_enso_r = {}


for key, df in combined_model_dfs.items():
    with localconverter(ro.default_converter + pandas2ri.converter):
        combined_models_r[key] = ro.conversion.py2rpy(df)
        combined_gmst_r[key] = ro.conversion.py2rpy(combined_gmst_dfs[key])
        combined_enso_r[key] = ro.conversion.py2rpy(combined_enso_dfs[key])

# Push to R environment
ro.globalenv["all_models_r"] = ListVector(combined_models_r)
ro.globalenv["all_gmst_r"] = ListVector(combined_gmst_r)
ro.globalenv["all_enso_r"] = ListVector(combined_enso_r)

# Call R functions
Process.analyze_extreme_scenario_2nd_cov()
analyze_extreme_scenario_2nd_cov = r['analyze_extreme_scenario_2nd_cov']

Process.merge_model_gmst()
merge_model_gmst = r['merge_model_gmst']


##### <font color='orange'> Model fitting </font>

Model fitting is performed three times (utilizing the same fit for both future scenarios), using each model’s smoothed GMST as a covariate. Please validate the selection of the number of bootstrap samples ($nsamp$) for uncertainty estimation, and the specific scenarios to be executed. Return period ($rp$) is automatically selected from 02_Trend_analysis results unless specified by the user.

**Note:** 
- For validation use the same years as in observations up to now
- In case the return period is very high (order 10,000 years), consider using the lower bound of the uncertainty range, or a standard return period of e.g. 100 or 500 years, depending on the uncertainty range. 
- In case the return period is close to 1 year (in the current climate), use a return period of 1.5 years: since we are using annual maxima/minima, the return period (and uncertainty range) cannot be lower than one year. Note that Probability Ratios for the future scenarios will not be informative.

In [ ]:
# Load the observational return period for the event
df_obs = pd.read_csv(f'{your_save_directory}/res-obs_era5.csv', index_col=0)
rp_obs = df_obs['return_period_est'].iloc[0]
print(f'Currently, the return period from observational data for the event is rp={rp_obs}')

In [ ]:
%%R -i your_save_directory -i rp_obs -i conf -i nino_obs_now -o dist

rp <- rp_obs  # Return Period (e.g., from Obs)

############################ User Selection ############################
nsamp <- 50 # Bootstrap samples for confidence intervals, use a larger number (e.g., 1000) for more robust results, but it will take longer to run
dist <- "gev" # Distribution for extreme value analysis (e.g., "gev", "norm"))
lower <- FALSE # Cold events: TRUE, Warm events: FALSE

scenarios <- list(
    list(ys=1950, ye=2025, yn=2025, dG_hist=1.3, dG_fut=0,   nino_hist=nino_obs_now, nino_fut=nino_obs_now,  lbl="Validation"),
    list(ys=1880, ye=2025, yn=2025, dG_hist=1.3, dG_fut=0,   nino_hist=nino_obs_now, nino_fut=nino_obs_now,  lbl="Pi"),
    list(ys=1880, ye=2025, yn=2025, dG_hist=0,   dG_fut=0, nino_hist=0, nino_fut=nino_obs_now, lbl="Neut"),
    list(ys=1880, ye=2025, yn=2025, dG_hist=1.3, dG_fut=0, nino_hist=0, nino_fut=nino_obs_now, lbl="Pineut"),
    list(ys=1880, ye=2100, yn=2025, dG_hist=1.3, dG_fut=0.7, nino_hist=nino_obs_now, nino_fut=nino_obs_now, lbl="Future-2.0"),
    list(ys=1880, ye=2100, yn=2025, dG_hist=1.3, dG_fut=1.3, nino_hist=nino_obs_now, nino_fut=nino_obs_now, lbl="Future-2.6")
)

# nino_obs_now  = observed current-year Niño (passed in from cov_observations.csv)
#            NA = resolves to nino_obs_now  → hold ENSO at observed conditions
#             0 = force ENSO-neutral conditions (nino = 0)

# ── dG_hist / dG_fut ────────────────────────────────────────────────────────
# 0   = no GMST shift from factual (i.e. remain in present climate)
# 1.3 = shift GMST by ±1.3 °C (subtracted for hist → pre-industrial,
#                                added for fut → projection)

########################################################################

# Setup
second_cov <- TRUE # Use second covariate (ENSO) in addition to GMST
library(rwwa)
library(extRemes)
all_scenarios_results <- list()
counter <- 1
# Loop through the combined list (contains both CORDEX and CMIP6)
for (name in names(all_models_r)) {
    cat("Processing Model:", name, "\n")
    
    m_df <- all_models_r[[name]]
    g_df <- all_gmst_r[[name]]
    e_gf <- all_enso_r[[name]]


    data_start_year <- min(m_df$year, na.rm = TRUE)
    is_recent_model <- data_start_year >= 1950

    validation_res <- NULL
    
    for (s in scenarios) {

        if (s$lbl == "Pi" && is_recent_model) {
            cat("   Skipping Pi scenario for", name, "(Data starts in", data_start_year, "Cloning validation results)\n")
            # Clone the validation result
            if (!is.null(validation_res)) {
                res_pi <- validation_res
                
                # Change the scenario label to "Pi"
                res_pi$scenario <- "Pi"
                
                # Append to your global tracking list
                all_scenarios_results[[counter]] <- res_pi
                counter <- counter + 1
            }
            next 
        }
        res <- analyze_extreme_scenario_2nd_cov(name, rp, m_df, g_df, e_gf, s$ys, s$ye, s$yn, nsamp, s$dG_hist, s$dG_fut, s$nino_hist, s$nino_fut, s$lbl, dist, conf, lower, your_save_directory, second_cov, nino_obs_now)
        
        if (!is.null(res)) {

            if (s$lbl == "Validation") {
                validation_res <- res
            }
            
            all_scenarios_results[[counter]] <- res
            counter <- counter + 1
        }
    }
}

# Save everything to ONE file
if (length(all_scenarios_results) > 0) {
    df_final <- do.call(rbind, all_scenarios_results)
    df_final$Include <- TRUE
    
    # Reorder columns: meta data first
    meta_cols <- c("Include", "model", "scenario")
    data_cols <- setdiff(names(df_final), meta_cols)
    df_final <- df_final[, c(meta_cols, data_cols)]

    out_file <- file.path(your_save_directory, "res-models-all.csv")
    write.csv(df_final, out_file, row.names = FALSE, quote = FALSE)
    
    cat("\n✅ Success! Saved all CMIP6 and CORDEX results to:", out_file, "\n")
} else {
    cat("\n❌ No results were generated. Check your data input.\n")
}

#### d.	Compare statistical fit parameters

Compare statistical fit parameters over a period as similar as possible to the observed data: for ‘now’, or spanning the same historical period (e.g. 1950-now). Check findings in the automated output table and combine with general properties into one conclusion.


##### <font color='green'>  Extract observational data </font>

In [ ]:
df_validation = pd.read_csv(os.path.join(your_save_directory, "res-models-all.csv"))

# Run the sync
active_params = Utils.extract_results(parameter, df_validation_results, df_validation, df_obs, dist, conf, second_cov=True)

with open(os.path.join(your_save_directory, "active-params.json"), 'w') as f:
    json.dump(active_params, f)

##### <font color='orange'> Take a decision over results </font>

Assess the statistical consistency between the model and observations using the fit parameters ($\sigma$ or $\sigma$/$\mu$  and $\xi$). Use the following criteria for inclusion (automated):
- i. Good: The best estimates of the model fit parameters fall within the observed confidence intervals.
- ii. Reasonable: The confidence intervals of the model and observations partially overlap (minimum 5% overlap);
- iii. Bad: The confidence intervals do not overlap; exclude these models from the synthesis.

Additionally, evaluate the magnitude difference at the specified return period to assess bias. Exclude model runs requiring excessive bias correction that results in an unphysical or significantly divergent event magnitude compared to observations.

In [ ]:
Utils.create_decision_hub(df_validation_results, step='statistics', project_filter='all', save_path=validation_file_path, active_params=active_params, second_cov=True)

###  4.6 Decide on which models to include 

Decide on which models to include in the model synthesis. Note that if we have enough “good” models we can discuss whether we want to do the synthesis without including the "reasonable" models. If we still use the "reasonable" ones this should be noted as a caution in the associated text in Scientific report Section 4 and potentially Section 6 if only “reasonable” models are included. 
The overall conclusion is “good” if all columns indicate “good", “reasonable” if at least one column indicates “reasonable” and “bad” if one column indicates “bad”. However, in the comments column one could indicate whether “reasonable” stems from only one “reasonable” or several, see next substep.  
- a. If, per framing/model setup we have five or more models (from a different model ensemble) that are “good” according to the validation we do not use the “reasonable” models for that framing/model setup but only the “good” models. If we have less than five “good” models, first add only models with one “reasonable”, if still less than five models add models with two “reasonable” labels, etc. 
- b. If we have two models stemming from the same model, e.g., a high medium and/or a lower resolution model (model names differ only in ‘_hr’, ‘_mr’ and ‘_lr’), check (by eye) if the results in the attribution step are the same. If so, only use one of the models, otherwise keep both. If one of the models performs better than the other (‘good’ versus ‘reasonable’) use the best model. 
- c. All decisions that deviate from the above points need to be documented including the reason of the decision in the Comments row.


### <font color='orange'> 4.7 State other considerations for including or excluding models </font>

Other considerations for including or excluding a model must be clearly documented, explicitly stating the decisions and motivation for the decision.

- a.	Write any consideration in the Comments row, or
- b.	If decisions are more general write them in the scientific report.
- c.	Make sure the “Include?” column in the table is final and filled with the final decisions on “True” (include model) or “False” (don’t include model).
- d.	Save output table model validation (res-models-validation.csv).
- e.	Save output table all results (res-models-final-decisions.csv).
- f.	Save output table past and future (res-models-Future-2.0.csv and res-models-Future-2.6.csv, standard res-models-Future-2.6.csv will be used in the synthesis).


In [ ]:
Utils.create_decision_hub(df_validation_results, step='full', project_filter='all', save_path=validation_file_path, active_params=active_params, second_cov=True)

##### <font color='orange'> Save outputs </font>
Run the cell below to combine all your decisions in the output file used for the next step (6. Synthesis)

In [ ]:
# Final decision map
decision_map = df_validation_results.set_index('model')['Include T/F'].to_dict()

# Map decisions and set index
df_validation_final = df_validation.copy()
df_validation_final['Include'] = df_validation_final['model'].map(decision_map)
df_validation_final.set_index('model', inplace=True)
df_validation_final.index.name = None

# Save the full file
full_path = os.path.join(your_save_directory, "res-models-final-decisions.csv")
df_validation_final.to_csv(full_path, index=True)

# Create and save the NEUT
df_neut = df_validation_final[df_validation_final['scenario'] == 'Neut']
df_neut = df_neut.drop(columns=['scenario'])
path_neut = os.path.join(your_save_directory, "res-models-Neut.csv")
df_neut.to_csv(path_neut, index=True)

# Create and save the PINEUT
df_pineut = df_validation_final[df_validation_final['scenario'] == 'Pineut']
df_pineut = df_pineut.drop(columns=['scenario'])
path_pineut = os.path.join(your_save_directory, "res-models-Pineut.csv")
df_pineut.to_csv(path_pineut, index=True)

# Create and save the Future-2.0 file
df_20 = df_validation_final[df_validation_final['scenario'] == 'Future-2.0']
df_20 = df_20.drop(columns=['scenario'])
path_20 = os.path.join(your_save_directory, "res-models-Future-2.0.csv")
df_20.to_csv(path_20, index=True)

# Create and save the Future-2.6 file
df_26 = df_validation_final[df_validation_final['scenario'] == 'Future-2.6']
df_26 = df_26.drop(columns=['scenario'])
path_26 = os.path.join(your_save_directory, "res-models-Future-2.6.csv")
df_26.to_csv(path_26, index=True)

print(f"✅ Three files saved successfully in: {your_save_directory}")
print(f"1. Full: {os.path.basename(full_path)}")
print(f"2. NEUT: {os.path.basename(path_neut)}")
print(f"3. PINEUT: {os.path.basename(path_pineut)}")
print(f"4. Future-2.0: {os.path.basename(path_20)}")
print(f"5. Future-2.6: {os.path.basename(path_26)}")

### <font color='green'> 5.6 Save and check all manual choices </font>

Check that all manual choices have been written in the validation table. Only add text to scientific report Sections 4 and 5 if important decisions do not fit into this table. 
- Save output table for Scientific report validation Table_4_1.csv
- Copy this table to the Scientific report Section 4.

In [ ]:
def validation_table(data_dir=your_save_directory): #the location of the csv files in the data folder 
    #loading the source csv files 
    validation_file_path = os.path.join(data_dir, "res-models-validation.csv")
    all_models_file_path = os.path.join(data_dir, "res-models-all.csv")
    era5_file_path  = os.path.join(data_dir, "res-obs_era5.csv")
    
    #if the csv files cannot be found, error message 
    for file in [validation_file_path,all_models_file_path, era5_file_path]:
        if not os.path.exists(file):
            print(f"Error: {file} not found in the data folder.")
            return
    
    #reading in the csv files 
    df_validation = pd.read_csv(validation_file_path)
    df_all = pd.read_csv(all_models_file_path)
    df_era5 = pd.read_csv(era5_file_path)
    
    
    #filter for only the validation scenario results in the res-models-all.csv files 
    #check the column names 
    df_all.columns
    df_all["scenario"].unique() 
    df_all_validation = df_all[df_all["scenario"] == "Validation"].copy()
    
    #initialising an empty list for the rows that are included in the final validation table 
    rows = []
    
    
    #### processing the ERA5 cata ####
    #use the data on the distribution and configuration from the first available model to decide which variables (sigma, shape, dispersion) to extract for ERA5
    first_model = df_all_validation.iloc[0]
    #getting information from user selection about the distribution (norm or gev) and the configuration (shift, fixeddisp)
    #this is given as an R output which needs to be converted to a Python string 
    dist_type = dist[0].strip().lower()
    conf_type = conf.strip().lower()
    
    #getting the correct unit of meaaurement for the event magnitude 
    current_unit = f"({unit})" #°C or mm
    
    era5_data = df_era5.iloc[0]
    era5_entry = {
        "Model" : "ERA5 (Obs)",
        "Seasonal cycle": "-", #blank for observations 
        "Spatial maps": "-", #blank for observations 
        f"Event magnitude {current_unit}": round(era5_data.get("event_magnitude_est", 0), 2),}
    
    #Statistical parameters 
    #if the configuration chosen is shift (temperature cases), sigma is reported 
    if conf_type == "shift":
        era5_entry[f"Sigma est {current_unit}"] = round(era5_data.get("sigma0_est", 0), 2)
        era5_entry[f"Sigma lower {current_unit}"] = round(era5_data.get("sigma0_lower", 0), 2)
        era5_entry[f"Sigma upper {current_unit}"] = round(era5_data.get("sigma0_upper", 0), 2)
        
    #if the configuration chosen is fixeddisp (scale) for precipitation cases, dispersion is reported
    elif conf_type == "fixeddisp":
        era5_entry["Dispersion est (-)"] = round(era5_data.get("disp_est", 0), 2) # the 0 is added as a safety default in case the column is missing, rounded to 2 decimal places 
        era5_entry["Dispersion lower (-)"] = round(era5_data.get("disp_lower", 0), 2)
        era5_entry["Dispersion upper (-)"] = round(era5_data.get("disp_upper", 0), 2)
     
    # if the distribution chosen is GEV, the shape parameter is added into the validation table 
    if dist_type == "gev":
        era5_entry["Shape est (-)"] = round(era5_data.get("shape_est", 0), 2)
        era5_entry["Shape lower (-)"] = round(era5_data.get("shape_lower", 0), 2)
        era5_entry["Shape upper (-)"] = round(era5_data.get("shape_upper", 0), 2)

    era5_entry["Nino corr est (-)"] = round(era5_data.get("nino_corr_est", 0), 2)
    era5_entry["Nino corr lower (-)"] = round(era5_data.get("nino_corr_lower", 0), 2)
    era5_entry["Nino corr upper (-)"] = round(era5_data.get("nino_corr_upper", 0), 2)  
        
    era5_entry["Final decision"] = "-" #not applicable for ERA5
    era5_entry["Comments"] = "-" #not applicable for ERA5
    rows.append(era5_entry)
    
    
    #### model validation ####
    
    #looping through the res-models-validation.csv file 
    for _, v_row in df_validation.iterrows(): #v_row represents a single row of the csv file 
                                              # _ is the placeholder for the inde number 
        model_name = v_row["model"] #fetch the name of the models from the current row 
        #looking into the res-models-all.csv file and look for the row where the model name matches the one above
        a_row = df_all_validation[df_all_validation["model"] == model_name]
        
        if a_row.empty:
            continue
        #retrieving the data from the first matching row
        a_row = a_row.iloc[0]
        
        
        #columns of the validation output table 
        entry = {
            "Model": model_name,
            "Seasonal cycle": v_row.get("Seasonal cycle", ""),
            "Spatial maps": v_row.get("Spatial maps", ""),
            #the event magnitude is labelled rp_value because the value is calculated for a specific return period 
            f"Event magnitude {current_unit}": round(a_row.get("rp_value", 0), 2),}
        
        #Statistical parameters 
        #if the configuration chosen is shift (temperature cases), sigma is reported 
        if conf_type == "shift":
            entry[f"Sigma est {current_unit}"] = round(a_row.get("eval_sigma0_est", 0), 2)
            entry[f"Sigma lower {current_unit}"] = round(a_row.get("eval_sigma0_lower", 0), 2)
            entry[f"Sigma upper {current_unit}"] = round(a_row.get("eval_sigma0_upper", 0), 2)
            
        #if the configuration chosen is fixeddisp (scale) for precipitation cases, dispersion is reported
        elif conf_type == "fixeddisp":
            entry["Dispersion est (-)"] = round(a_row.get("eval_disp_est", 0), 2) # the 0 is added as a safety default in case the column is missing, rounded to 2 decimal places 
            entry["Dispersion lower (-)"] = round(a_row.get("eval_disp_lower", 0), 2)
            entry["Dispersion upper (-)"] = round(a_row.get("eval_disp_upper", 0), 2)
         
        # if the distribution chosen is GEV, the shape parameter is added into the validation table 
        if dist_type == "gev":
            entry["Shape est (-)"] = round(a_row.get("eval_shape_est", 0), 2)
            entry["Shape lower (-)"] = round(a_row.get("eval_shape_lower", 0), 2)
            entry["Shape upper (-)"] = round(a_row.get("eval_shape_upper", 0), 2)

        entry["Nino corr est (-)"] = round(a_row.get("nino_corr_est", 0), 2)
        entry["Nino corr lower (-)"] = round(a_row.get("nino_corr_lower", 0), 2)
        entry["Nino corr upper (-)"] = round(a_row.get("nino_corr_upper", 0), 2)  
        #addin the final decision and the notes
        entry["Final decision"] = v_row.get("Include T/F", "")
        entry["Comments"] = v_row.get("Comments", "")
        
        rows.append(entry)
        
    #exporting to a df
    df_final = pd.DataFrame(rows)
    
    #reordering the columns 
    columns = ["Model", "Seasonal cycle", "Spatial maps"]
    #dynamically finding the statistical parameters based on the chosen distribution and configureation 
    #looking for columns that contain sigma, dispersion, and shape 
    parameters_column = [c for c in df_final.columns if any(x in c for x in ["Sigma", "Dispersion", "Shape", "Nino"])]
    final_columns = columns + parameters_column + [f"Event magnitude {current_unit}", "Final decision", "Comments"] 
    
    df_final = df_final[final_columns]
    
    #saving the csv file as Table_4_1.csv
    output_file = os.path.join(data_dir, "Table_4_1.csv")
    df_final.to_csv(output_file, index=False)
    
    return df_final
    
    
#execute the function 
df_table = validation_table()

#show the table 
df_table